<a href="https://colab.research.google.com/github/kreaja/TrialRepositoryJustForMe/blob/main/hw04_cis5300.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW04: Word Embeddings
## CIS 5300: Natural Language Processing | Module 04 | 100 Points

---

### Overview

In this assignment, you will explore two paradigms for representing word meaning as vectors: **count-based representations** (term-document matrices, TF-IDF, PPMI) and **dense embeddings** (Word2Vec, GloVe). You will build count-based representations from a Shakespeare corpus, load pre-trained dense embeddings, compare the two approaches on word similarity benchmarks, investigate bias in embeddings, and apply both representations to a word sense clustering task.

The assignment mirrors the arc of Module 04: starting with sparse, interpretable count-based vectors and progressing to dense, learned representations. By the end, you will have firsthand experience with why dense embeddings largely replaced count-based methods -- and where they still fall short.

This homework connects directly to the Module 04 lectures:
- **Word Meaning & Similarity:** Word Meaning and Contrast through Connotations and Distributional Similarity
- **Count-Based Representations:** Word Representation by Documents and by Context
- **Dense Embeddings (Word2Vec):** Introducing Word2Vec through Evaluating Embeddings
- **Analogies & Vector Arithmetic:** Analogies and Vector Arithmetic
- **Bias in Embeddings:** Bias in Embeddings
- **Context Dependence & Limitations:** Context Dependence

**Textbook Reference:** [Chapter 5: Vector Semantics and Embeddings](https://web.stanford.edu/~jurafsky/slp3/5.pdf) of Speech and Language Processing by Jurafsky & Martin.

### Point Breakdown

| Section | Topic | Points | Type |
|---------|-------|--------|------|
| 0 | Setup | 0 | -- |
| 1 | Count-Based Representations | 25 | Code |
| 2 | Dense Embeddings (incl. polysemy & analogy failures) | 25 | Code + Writeup |
| 3 | Word Sense Clustering | 25 | Code + Writeup |
| 4 | Analysis & Reflection | 25 | Writeup |
| **Total** | | **100** | |

### Deliverables

Submit the following to Gradescope:

| File                  | Description                                           |
|-----------------------|-------------------------------------------------------|
| `hw04.ipynb` | This notebook with all code cells completed           |
| `hw04.py`             | Code from this notebook, downloaded as a .py file     |
| `hw04_writeup.pdf`          | Written answers to all questions marked **[WRITEUP]** |

### Recommended Reading

- [Chapter 5: Vector Semantics and Embeddings](https://web.stanford.edu/~jurafsky/slp3/5.pdf) -- Jurafsky & Martin, SLP 3rd Edition
- [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781) -- Mikolov et al., 2013 (Word2Vec)
- [GloVe: Global Vectors for Word Representation](https://nlp.stanford.edu/pubs/glove.pdf) -- Pennington et al., 2014
- [Man is to Computer Programmer as Woman is to Homemaker?](https://arxiv.org/abs/1607.06520) -- Bolukbasi et al., 2016 (Bias)
- Module 04 lectures (Word Meaning and Contrast through Context Dependence)

---
## Section 0: Setup (0 points)

Run the following cells to install packages and load the data. **Do not modify these cells** unless you need to adjust the embedding download path.

In [1]:
# Install required packages
# Cell 3
import sys
# %pip install numpy scikit-learn==1.6.1 gensim matplotlib huggingface-hub
sys.print(__version__)
# What version of every package should I use to avoid compatibility issues (this notebook is a little bit old)?


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 28.7 MB/s eta 0:00:00


In [2]:
# === Data Download ===
# Cell 4
# Downloads homework data from HuggingFace.
# This uses the huggingface_hub library to fetch the data files.
# Files are cached locally so re-running this cell is instant.
import os
try:
    if not os.path.exists('data'):
        %pip install -q huggingface_hub
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id="CCB/cis5300-word-embeddings",
            repo_type="dataset",
            local_dir=".",
            allow_patterns="data/*"
        )
        print(f"Downloaded {len(os.listdir('data'))} files to data/")
    else:
        print("Data already downloaded.")
except Exception as e:
    print(f"[Data download skipped: {e}]")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

Downloaded 3 files to data/


In [3]:
# cell 5
import numpy as np
import os
import math
from collections import Counter, defaultdict
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# We will import gensim in Section 2 when we load pre-trained embeddings.

print("All imports successful.")

All imports successful.


### 0.1 Load Shakespeare Corpus

We use a collection of Shakespeare plays for count-based representations. Each play is stored as a separate text file in `data/shakespeare/`. The corpus is pre-tokenized (lowercased, punctuation removed) so you can focus on the vector space methods rather than text processing.

In [4]:
########## DO NOT MODIFY ##########

SHAKESPEARE_DIR = 'data/shakespeare'

try:
    assert os.path.isdir(SHAKESPEARE_DIR), \
        f"Missing directory: {SHAKESPEARE_DIR}. Make sure the data/ directory is present."

    # Load all plays: dict mapping play_name -> list of words
    plays = {}
    for filename in sorted(os.listdir(SHAKESPEARE_DIR)):
        if filename.endswith('.txt'):
            play_name = filename.replace('.txt', '').replace('_', ' ').title()
            with open(os.path.join(SHAKESPEARE_DIR, filename), encoding='utf-8') as f:
                plays[play_name] = f.read().lower().split()

    play_names = sorted(plays.keys())
    print(f"Loaded {len(plays)} plays.")
    print(f"Play names: {play_names[:5]} ... ({len(play_names)} total)")
    print(f"Example: '{play_names[0]}' has {len(plays[play_names[0]]):,} words")

    # Build a combined corpus for word co-occurrence
    all_words = []
    for name in play_names:
        all_words.extend(plays[name])
    print(f"\nTotal words across all plays: {len(all_words):,}")

    # Build vocabulary: words appearing at least 50 times
    MIN_COUNT = 50
    word_counts = Counter(all_words)
    vocab = sorted([w for w, c in word_counts.items() if c >= MIN_COUNT])
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    print(f"Vocabulary size (min count {MIN_COUNT}): {len(vocab)}")
    print(f"Sample vocabulary: {vocab[:10]}")

except Exception as e:
    print(f"[Shakespeare data loading skipped: {e}]")
    plays = {}
    play_names = []
    all_words = []
    vocab = []
    word_to_idx = {}

Loaded 12 plays.
Play names: ['A Midsummer Nights Dream', 'As You Like It', 'Hamlet', 'Julius Caesar', 'King Lear'] ... (12 total)
Example: 'A Midsummer Nights Dream' has 17,135 words

Total words across all plays: 280,937
Vocabulary size (min count 50): 674
Sample vocabulary: ['a', 'about', 'act', 'after', 'again', 'against', 'age', 'air', 'alas', 'albany']


### 0.2 Load Pre-trained Embeddings

We use **gensim** to load pre-trained word embeddings.

**Recommendation:** while you're developing your code, use the smaller GloVe model so iteration is fast (~128 MB, loads in roughly 30 seconds). Once your code works, switch to the larger Word2Vec Google News model for the final analyses you write up in your report. The full Word2Vec download is **1.5 GB and takes 15-20 minutes the first time**, so don't reach for it until you actually need it.

**Default — GloVe 6B 100d (~128 MB):**
```python
import gensim.downloader as api
embeddings = api.load('glove-wiki-gigaword-100')
```

**Final pass — Word2Vec Google News 300d (1.5 GB):**
```python
import gensim.downloader as api
embeddings = api.load('word2vec-google-news-300')
```

**If both downloads fail:** Download GloVe manually from https://nlp.stanford.edu/data/glove.6B.zip, unzip, and load:
```python
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec
glove2word2vec('glove.6B.100d.txt', 'glove.6B.100d.w2v.txt')
embeddings = KeyedVectors.load_word2vec_format('glove.6B.100d.w2v.txt')
```

The rest of the notebook refers to the variable `embeddings`.

> **Note on writeup questions:** If you use the smaller **100d GloVe** model during development, some write-up results may look different from what the **300d Word2Vec** model produces -- in particular, the capitalization trick in **Answer 2.1b** (e.g., `apple` vs `Apple`) and the Kendall's Tau values in **Answer 2.3**. **When writing your report, use the 300d model if possible.** If you cannot (e.g., due to download or memory constraints), answer honestly based on what you observe with the 100d model -- partial or subtler results are fine as long as your analysis is accurate.

In [5]:
# Load pre-trained embeddings.
# Cell 9
# While developing your code, keep this set to the smaller GloVe model -- it
# loads in ~30 seconds and is plenty for testing your implementation. When
# you're ready to write up your report and want richer 300d vectors trained
# on much more text, switch to 'word2vec-google-news-300' (1.5 GB, 15-20 min
# the first time it downloads).
EMBEDDING_MODEL = 'glove-wiki-gigaword-100'   # change to 'word2vec-google-news-300' for the final pass

try:
    import gensim.downloader as api
    print(f"Loading pre-trained embeddings ({EMBEDDING_MODEL}) -- this may take a few minutes the first time...")
    embeddings = api.load(EMBEDDING_MODEL)
    print(f"Embedding vocabulary size: {len(embeddings):,}")
    print(f"Embedding dimensionality: {embeddings.vector_size}")
    print(f"Type: {type(embeddings)}")
except Exception as e:
    print(f"[Embeddings loading skipped: {e}]")
    embeddings = None

Loading pre-trained embeddings (glove-wiki-gigaword-100) -- this may take a few minutes the first time...
[==================================================] 100.0% 128.1/128.1MB downloaded
Embedding vocabulary size: 400,000
Embedding dimensionality: 100
Type: <class 'gensim.models.keyedvectors.KeyedVectors'>


### 0.3 Load Paraphrase Clustering Data

For Section 3, we load a dataset of paraphrases grouped by target word. Each target word has multiple paraphrase sets that correspond to different word senses. Your task will be to cluster the paraphrases to discover these senses.

**Data format:**
- `dev_data`: For development/tuning (gold-standard K and sense labels provided)
- `test_data`: For final evaluation (gold-standard K may or may not be provided)

In [6]:
########## DO NOT MODIFY ##########

CLUSTER_DEV_PATH = 'data/clustering/dev.txt'
CLUSTER_TEST_PATH = 'data/clustering/test.txt'
CLUSTER_COOCCURRENCE_DIR = 'data/clustering/cooccurrence'


def load_cluster_data(filepath):
    """Load paraphrase clustering data.

    Returns a list of dicts, each with:
        'target': the target word (string)
        'k': number of gold-standard senses (int)
        'paraphrases': list of paraphrase strings
        'labels': list of gold-standard cluster labels (ints)
    """
    data = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            target = parts[0]
            k = int(parts[1])
            paraphrases = []
            labels = []
            for item in parts[2:]:
                # Format: "paraphrase::label"
                pp, label = item.rsplit('::', 1)
                paraphrases.append(pp)
                labels.append(int(label))
            data.append({
                'target': target,
                'k': k,
                'paraphrases': paraphrases,
                'labels': labels
            })
    return data


try:
    for path in [CLUSTER_DEV_PATH, CLUSTER_TEST_PATH]:
        assert os.path.exists(path), f"Missing data file: {path}"
    assert os.path.isdir(CLUSTER_COOCCURRENCE_DIR), \
        f"Missing directory: {CLUSTER_COOCCURRENCE_DIR}"

    dev_data = load_cluster_data(CLUSTER_DEV_PATH)
    test_data = load_cluster_data(CLUSTER_TEST_PATH)

    print(f"Dev set: {len(dev_data)} target words")
    print(f"Test set: {len(test_data)} target words")
    print(f"\nExample entry:")
    ex = dev_data[0]
    print(f"  Target word: '{ex['target']}'")
    print(f"  Number of senses (K): {ex['k']}")
    print(f"  Paraphrases: {ex['paraphrases'][:5]} ...")
    print(f"  Gold labels: {ex['labels'][:5]} ...")

except Exception as e:
    print(f"[Clustering data loading skipped: {e}]")
    dev_data = []
    test_data = []

Dev set: 10 target words
Test set: 10 target words

Example entry:
  Target word: 'bank'
  Number of senses (K): 2
  Paraphrases: ['financial institution', 'savings account', 'investment firm', 'credit union', 'money lender'] ...
  Gold labels: [0, 0, 0, 0, 0] ...


---
## Section 1: Count-Based Representations (25 points)

In this section, you will build the classic count-based word vectors discussed in the Module 4 lectures on count-based representations (Word Representation by Documents and by Context) and Chapter 5, Sections 5.1-5.4 of the textbook. These methods represent words as high-dimensional sparse vectors based on their co-occurrence patterns with other words or documents.

**Key idea (the distributional hypothesis):** "You shall know a word by the company it keeps" (J.R. Firth, 1957). Words that appear in similar contexts tend to have similar meanings.

### 1.1 Term-Document Matrix (6 points)

Build a **term-document matrix** where rows are vocabulary words and columns are Shakespeare plays. Each cell contains the frequency of word $w$ in play $d$.

This gives each play a vector representation (a column) and each word a vector representation (a row). Two plays with similar word usage will have similar column vectors; two words that appear in the same plays will have similar row vectors.

After building the matrix, compute cosine similarity between plays and rank them by similarity to a target play.

**Lecture:** Word Representation by Documents | **Textbook:** Chapter 5, Section 5.3

In [7]:
# cell 14
def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors.

    cos(v1, v2) = (v1 . v2) / (||v1|| * ||v2||)

    Args:
        v1: numpy array of shape (d,)
        v2: numpy array of shape (d,)

    Returns:
        A float in [-1, 1]. Returns 0.0 if either vector has zero norm.
    """
    magns = None
    if np.linalg.norm(v1) != 0 and np.linalg.norm(v2) != 0:
        magns = np.linalg.norm(v1) * np.linalg.norm(v2)  #  guard against zero denominator
    else:
        return 0.0
    return np.dot(v1, v2) / magns

In [8]:
# Cell 15
try:
    # Sanity checks for cosine_similarity

    # Identical vectors -> 1.0
    v = np.array([1.0, 2.0, 3.0])
    assert abs(cosine_similarity(v, v) - 1.0) < 1e-10, \
        f"Cosine similarity of identical vectors should be 1.0, got {cosine_similarity(v, v)}"

    # Orthogonal vectors -> 0.0
    v1 = np.array([1.0, 0.0])
    v2 = np.array([0.0, 1.0])
    assert abs(cosine_similarity(v1, v2) - 0.0) < 1e-10, \
        f"Cosine similarity of orthogonal vectors should be 0.0, got {cosine_similarity(v1, v2)}"

    # Opposite vectors -> -1.0
    v1 = np.array([1.0, 2.0])
    v2 = np.array([-1.0, -2.0])
    assert abs(cosine_similarity(v1, v2) - (-1.0)) < 1e-10, \
        f"Cosine similarity of opposite vectors should be -1.0, got {cosine_similarity(v1, v2)}"

    # Zero vector -> 0.0
    v1 = np.array([1.0, 2.0])
    v2 = np.array([0.0, 0.0])
    assert cosine_similarity(v1, v2) == 0.0, \
        f"Cosine similarity with zero vector should be 0.0, got {cosine_similarity(v1, v2)}"

    # Known value
    v1 = np.array([1.0, 1.0])
    v2 = np.array([1.0, 0.0])
    expected = 1.0 / math.sqrt(2)
    assert abs(cosine_similarity(v1, v2) - expected) < 1e-10, \
        f"cos([1,1], [1,0]) should be 1/sqrt(2) = {expected:.4f}, got {cosine_similarity(v1, v2):.4f}"

    print("Cosine similarity sanity checks passed!")
except Exception as e:
    print(f"[Cosine similarity checks skipped: {e}]")

Cosine similarity sanity checks passed!


In [9]:
# cell 16

def build_term_document_matrix(plays, vocab, word_to_idx, play_names):
    """Build a term-document matrix from Shakespeare plays.

    Args:
        plays: Dict mapping play names to word lists.
        vocab: Sorted list of vocabulary words.
        word_to_idx: Dict mapping word to its index in vocab.
        play_names: Sorted list of play names.

    Returns:
        A numpy array of shape (len(vocab), len(play_names)).
        Entry [i, j] = number of times vocab[i] appears in play_names[j].
    """
    # Each play is one *column*
    occurrences_plays = np.array(len(vocab), len(play_names))
    # Keys will be two-element tuples, values integers
    for j, play in enumerate(play_names):
        count = Counter()
        count.update(plays[play])
        for word in count:
            if word not in vocab:  # vocab only contains words that appear >= 50
                continue       # times in shakespeare, so handle OOV words
            occurrences_plays[word_to_idx[word], j] = count[word]
            # occurrence_plays is a matrix with one row for each vocab word
    return occurrences_plays


In [10]:
# cell 17
# ~~~ Here - DWN DN
try:
    # Build the term-document matrix
    td_matrix = build_term_document_matrix(plays, vocab, word_to_idx, play_names)

    assert td_matrix.shape == (len(vocab), len(play_names)), \
        f"Expected shape ({len(vocab)}, {len(play_names)}), got {td_matrix.shape}"
    assert td_matrix.dtype in [np.float64, np.int64, np.float32, np.int32], \
        f"Matrix should be numeric, got dtype {td_matrix.dtype}"
    assert td_matrix.min() >= 0, \
        "All counts should be non-negative"
    assert td_matrix.sum() > 0, \
        "Matrix should not be all zeros"

    print(f"Term-document matrix shape: {td_matrix.shape}")
    print(f"Total word occurrences captured: {int(td_matrix.sum()):,}")
    print(f"Sparsity: {100 * (td_matrix == 0).sum() / td_matrix.size:.1f}% zeros")
except Exception as e:
    print(f"[Term-document matrix skipped: {e}]")
    td_matrix = None

[Term-document matrix skipped: Cannot interpret '12' as a data type]


In [11]:
# cell 18
def rank_plays_by_similarity(td_matrix, target_play, play_names):
    """Rank all plays by cosine similarity to a target play.

    Use the COLUMN vectors of the term-document matrix as play representations.

    Args:
        td_matrix: numpy array of shape (|V|, |D|).
        target_play: Name of the play to compare against (string).
        play_names: Sorted list of play names.

    Returns:
        A list of (play_name, similarity) tuples, sorted by similarity
        in descending order. Excludes the target play itself.
    """
    # Compute cosine similarity between two vectors:
    # cosine_similarity(v1, v2):
    similarities: list[str, float] = []
    for play in play_names:
        if play != target_play:
            continue
        cs: float = cosine_similarity(target_play, play)
        similarities.append((play, cs))
    return similarities

In [12]:
# Cell 19
try:
    # Rank plays by similarity to a target
    target = play_names[0]  # Use the first play alphabetically
    rankings = rank_plays_by_similarity(td_matrix, target, play_names)

    assert len(rankings) == len(play_names) - 1, \
        f"Should return {len(play_names)-1} plays (excluding target), got {len(rankings)}"
    assert all(isinstance(r, tuple) and len(r) == 2 for r in rankings), \
        "Each ranking should be a (play_name, similarity) tuple"
    assert rankings[0][1] >= rankings[-1][1], \
        "Rankings should be sorted by similarity in descending order"

    print(f"Plays most similar to '{target}':")
    for name, sim in rankings[:5]:
        print(f"  {name:<35} similarity = {sim:.4f}")
    print(f"\nPlays least similar to '{target}':")
    for name, sim in rankings[-3:]:
        print(f"  {name:<35} similarity = {sim:.4f}")
except Exception as e:
    print(f"[Play ranking skipped: {e}]")

[Play ranking skipped: could not convert string to float: np.str_('A Midsummer Nights Dream')]


### 1.2 Term-Context Matrix (6 points)

Build a **word co-occurrence matrix** where both rows and columns are vocabulary words. Cell $(i, j)$ counts how many times word $j$ appears within a context window of size $w$ around word $i$ in the corpus.

For a window size of $w$, the context of a word at position $t$ consists of the $w$ words before it and the $w$ words after it (positions $t-w$ through $t+w$, excluding $t$ itself).

**Example** with window $w=2$ and sentence "the cat sat on the mat":
- Context of "sat" (position 2): ["the", "cat", "on", "the"]
- Context of "cat" (position 1): ["the", "sat", "on"]

After building the matrix, explore how window size affects which words are most similar.

**Lectures:** Word Representation by Context, Varying Context Windows | **Textbook:** Chapter 5, Section 5.3

In [13]:
# cell 21
def build_cooccurrence_matrix(all_words, vocab, word_to_idx, window=2):
    """Build a word co-occurrence matrix.

    Args:
        all_words: List of all words in the corpus (in order).
        vocab: Sorted list of vocabulary words.
        word_to_idx: Dict mapping word to index in vocab.
        window: Context window size (number of words on each side).

    Returns:
        A numpy array of shape (len(vocab), len(vocab)).
        Entry [i, j] = number of times vocab[j] appears within
        a window of 'window' words around vocab[i] in the corpus.
    """
    # build_cooccurrence_matrix(all_words, vocab, word_to_idx, window=2)
    # for w in vocab:
    #   count = Counter()
    #   for (time w occurs in all_words):
    #     times = (num. times w2 is present within `window` of w)
    #   count[w, ] += times
    pass

In [ ]:
try:
    # Build co-occurrence matrix with window=2
    cooc_matrix = build_cooccurrence_matrix(all_words, vocab, word_to_idx, window=2)

    assert cooc_matrix.shape == (len(vocab), len(vocab)), \
        f"Expected shape ({len(vocab)}, {len(vocab)}), got {cooc_matrix.shape}"
    assert cooc_matrix.min() >= 0, \
        "All counts should be non-negative"
    # Co-occurrence should be symmetric (word j in context of i implies i in context of j)
    assert np.allclose(cooc_matrix, cooc_matrix.T, atol=1), \
        "Co-occurrence matrix should be approximately symmetric"

    print(f"Co-occurrence matrix shape: {cooc_matrix.shape}")
    print(f"Sparsity: {100 * (cooc_matrix == 0).sum() / cooc_matrix.size:.1f}% zeros")
except Exception as e:
    print(f"[Co-occurrence matrix skipped: {e}]")
    cooc_matrix = None

[Co-occurrence matrix skipped: 'NoneType' object has no attribute 'shape']


In [ ]:
def most_similar_cooc(word, cooc_matrix, vocab, word_to_idx, top_n=10):
    """Find the most similar words using co-occurrence vectors.

    Args:
        word: The query word (string).
        cooc_matrix: Co-occurrence matrix of shape (|V|, |V|).
        vocab: Sorted list of vocabulary words.
        word_to_idx: Dict mapping word to index.
        top_n: Number of results to return.

    Returns:
        A list of (word, similarity) tuples, sorted by similarity descending.
        Excludes the query word itself.
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Explore: how does window size affect similarity?
    test_word = 'king'  # Change this to explore other words

    for w in [1, 2, 5, 10]:
        cooc_w = build_cooccurrence_matrix(all_words, vocab, word_to_idx, window=w)
        similar = most_similar_cooc(test_word, cooc_w, vocab, word_to_idx, top_n=10)
        print(f"\nWindow = {w}: most similar to '{test_word}'")
        for w_name, sim in similar:
            print(f"  {w_name:<20} {sim:.4f}")
except Exception as e:
    print(f"[Window size exploration skipped: {e}]")


Window = 1: most similar to 'king'
[Window size exploration skipped: 'NoneType' object is not iterable]


### 1.3 TF-IDF Weighting (7 points)

Raw term frequencies overweight common words like "the" and "and". **TF-IDF** (Term Frequency -- Inverse Document Frequency) addresses this by downweighting words that appear in many documents.

For word $w$ in document $d$:

$$\text{tf-idf}(w, d) = \text{tf}(w, d) \times \text{idf}(w)$$

where:
- $\text{tf}(w, d)$ = frequency of $w$ in $d$ (raw count from your term-document matrix)
- $\text{idf}(w) = \log_{10}\left(\frac{N}{\text{df}(w)}\right)$
- $N$ = total number of documents
- $\text{df}(w)$ = number of documents containing $w$

**Lecture:** Word Representation by Documents | **Textbook:** Chapter 5, Section 5.3

In [ ]:
def compute_tfidf(td_matrix):
    """Compute TF-IDF weighting of a term-document matrix.

    Args:
        td_matrix: numpy array of shape (|V|, |D|) with raw term frequencies.

    Returns:
        A numpy array of the same shape with TF-IDF weighted values.
        Use log base 10 for the IDF computation.
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Compute TF-IDF matrix
    tfidf_matrix = compute_tfidf(td_matrix)

    assert tfidf_matrix.shape == td_matrix.shape, \
        f"TF-IDF matrix should have same shape as input, got {tfidf_matrix.shape}"
    assert tfidf_matrix.min() >= 0, \
        "TF-IDF values should be non-negative"

    # Words appearing in ALL documents should have IDF=0 -> TF-IDF=0
    appears_everywhere = (td_matrix > 0).all(axis=1)
    if appears_everywhere.any():
        assert (tfidf_matrix[appears_everywhere] == 0).all(), \
            "Words appearing in every document should have TF-IDF = 0 (IDF = log(N/N) = 0)"
        print(f"Verified: {appears_everywhere.sum()} words appearing in all plays have TF-IDF = 0")

    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    print(f"TF-IDF range: [{tfidf_matrix.min():.4f}, {tfidf_matrix.max():.4f}]")
except Exception as e:
    print(f"[TF-IDF matrix skipped: {e}]")
    tfidf_matrix = None

[TF-IDF matrix skipped: 'NoneType' object has no attribute 'shape']


In [ ]:
try:
    # Compare raw frequency vs TF-IDF: most discriminative words per play
    target_play_idx = play_names.index(play_names[0])

    print(f"Most frequent words in '{play_names[0]}' (raw counts):")
    raw_col = td_matrix[:, target_play_idx]
    top_raw = np.argsort(raw_col)[::-1][:10]
    for idx in top_raw:
        print(f"  {vocab[idx]:<20} count = {int(raw_col[idx])}")

    print(f"\nMost discriminative words in '{play_names[0]}' (TF-IDF):")
    tfidf_col = tfidf_matrix[:, target_play_idx]
    top_tfidf = np.argsort(tfidf_col)[::-1][:10]
    for idx in top_tfidf:
        print(f"  {vocab[idx]:<20} TF-IDF = {tfidf_col[idx]:.4f}")
except Exception as e:
    print(f"[TF-IDF comparison skipped: {e}]")

Most frequent words in 'A Midsummer Nights Dream' (raw counts):
[TF-IDF comparison skipped: 'NoneType' object is not subscriptable]


### 1.4 PPMI Weighting (6 points)

**Positive Pointwise Mutual Information (PPMI)** is the standard weighting for word co-occurrence matrices. It measures whether two words co-occur more than expected by chance.

For a co-occurrence matrix $F$ where $f_{ij}$ is the count of word $i$ appearing in the context of word $j$:

$$\text{PMI}(w, c) = \log_2 \frac{P(w, c)}{P(w) \cdot P(c)}$$

where:
- $P(w, c) = \frac{f_{wc}}{\sum_{i,j} f_{ij}}$ (joint probability)
- $P(w) = \frac{\sum_j f_{wj}}{\sum_{i,j} f_{ij}}$ (marginal probability of word)
- $P(c) = \frac{\sum_i f_{ic}}{\sum_{i,j} f_{ij}}$ (marginal probability of context)

PPMI clips negative values to zero:

$$\text{PPMI}(w, c) = \max(0, \text{PMI}(w, c))$$

**Handling zeros:** If $f_{wc} = 0$, then $P(w, c) = 0$ and $\log_2(0)$ is undefined. In this case, set $\text{PMI}(w, c) = 0$, which gives $\text{PPMI}(w, c) = 0$. This is consistent with the intuition that words that never co-occur provide no positive evidence of association.

**Lecture:** Word Representation by Context | **Textbook:** Chapter 5, Section 5.3

#### Worked Example: PPMI Step by Step

Consider this tiny co-occurrence matrix with 3 words:

|        | **cat** | **dog** | **fish** |
|--------|---------|---------|----------|
| **cat**  | 0       | 8       | 2        |
| **dog**  | 8       | 0       | 1        |
| **fish** | 2       | 1       | 0        |

**Step 1: Compute totals.**
- Total sum $S = 0 + 8 + 2 + 8 + 0 + 1 + 2 + 1 + 0 = 22$
- Row sums: cat = 10, dog = 9, fish = 3
- Column sums: cat = 10, dog = 9, fish = 3

**Step 2: Compute probabilities for PMI(cat, dog).**
- $P(\text{cat}, \text{dog}) = 8/22 = 0.3636$
- $P(\text{cat}) = 10/22 = 0.4545$
- $P(\text{dog}) = 9/22 = 0.4091$
- $\text{PMI} = \log_2(0.3636 / (0.4545 \times 0.4091)) = \log_2(1.956) = 0.968$
- $\text{PPMI} = \max(0, 0.968) = 0.968$

**Step 3: Compute PMI(cat, cat).**
- $P(\text{cat}, \text{cat}) = 0/22 = 0$ (zero count!)
- Since $P(w, c) = 0$, we set $\text{PMI} = 0$ and $\text{PPMI} = 0$.

**Step 4: Compute PMI(dog, fish).**
- $P(\text{dog}, \text{fish}) = 1/22 = 0.0455$
- $P(\text{dog}) = 9/22 = 0.4091$
- $P(\text{fish}) = 3/22 = 0.1364$
- $\text{PMI} = \log_2(0.0455 / (0.4091 \times 0.1364)) = \log_2(0.815) = -0.295$
- $\text{PPMI} = \max(0, -0.295) = 0$ (negative PMI is clipped!)

In [ ]:
def compute_ppmi(cooc_matrix):
    """Compute PPMI weighting of a co-occurrence matrix.

    For each cell (w, c):
        PMI(w, c) = log2(P(w,c) / (P(w) * P(c)))
        PPMI(w, c) = max(0, PMI(w, c))

    When the co-occurrence count is 0, set PMI = 0 (which gives PPMI = 0).
    Use log base 2.

    Args:
        cooc_matrix: numpy array of shape (|V|, |V|) with co-occurrence counts.

    Returns:
        A numpy array of the same shape with PPMI values.
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Verify with the worked example
    example_matrix = np.array([[0, 8, 2],
                                [8, 0, 1],
                                [2, 1, 0]], dtype=float)

    ppmi_example = compute_ppmi(example_matrix)

    # PMI(cat, dog) should be ~0.968
    expected_cat_dog = math.log2(8/22 / (10/22 * 9/22))
    assert abs(ppmi_example[0, 1] - expected_cat_dog) < 0.01, \
        f"PPMI(cat, dog) should be ~{expected_cat_dog:.3f}, got {ppmi_example[0, 1]:.3f}"

    # PMI(cat, cat) should be 0 (zero count)
    assert ppmi_example[0, 0] == 0.0, \
        f"PPMI(cat, cat) should be 0.0 (zero co-occurrence), got {ppmi_example[0, 0]}"

    # PMI(dog, fish) should be 0 (negative PMI clipped)
    assert ppmi_example[1, 2] == 0.0, \
        f"PPMI(dog, fish) should be 0.0 (negative PMI clipped), got {ppmi_example[1, 2]}"

    # All PPMI values should be >= 0
    assert ppmi_example.min() >= 0.0, \
        f"All PPMI values should be >= 0 (that's the 'positive' in PPMI), got min = {ppmi_example.min()}"

    print("Worked example PPMI values:")
    labels = ['cat', 'dog', 'fish']
    for i in range(3):
        for j in range(3):
            print(f"  PPMI({labels[i]}, {labels[j]}) = {ppmi_example[i, j]:.3f}")

    print("\nPPMI worked example checks passed!")
except Exception as e:
    print(f"[PPMI worked example skipped: {e}]")

[PPMI worked example skipped: 'NoneType' object is not subscriptable]


In [ ]:
try:
    # Compute PPMI on the full Shakespeare co-occurrence matrix
    ppmi_matrix = compute_ppmi(cooc_matrix)

    assert ppmi_matrix.shape == cooc_matrix.shape
    assert ppmi_matrix.min() >= 0.0, "PPMI values should be non-negative"

    print(f"PPMI matrix shape: {ppmi_matrix.shape}")
    print(f"PPMI range: [{ppmi_matrix.min():.4f}, {ppmi_matrix.max():.4f}]")
    print(f"Sparsity: {100 * (ppmi_matrix == 0).sum() / ppmi_matrix.size:.1f}% zeros")
except Exception as e:
    print(f"[PPMI matrix skipped: {e}]")
    ppmi_matrix = None

[PPMI matrix skipped: 'NoneType' object has no attribute 'shape']


In [ ]:
try:
    # Compare PPMI-weighted vs raw co-occurrence word similarity
    test_word = 'king'

    print(f"Most similar to '{test_word}' (raw co-occurrence):")
    similar_raw = most_similar_cooc(test_word, cooc_matrix, vocab, word_to_idx, top_n=10)
    for w, sim in similar_raw:
        print(f"  {w:<20} {sim:.4f}")

    print(f"\nMost similar to '{test_word}' (PPMI-weighted):")
    similar_ppmi = most_similar_cooc(test_word, ppmi_matrix, vocab, word_to_idx, top_n=10)
    for w, sim in similar_ppmi:
        print(f"  {w:<20} {sim:.4f}")
except Exception as e:
    print(f"[PPMI comparison skipped: {e}]")

Most similar to 'king' (raw co-occurrence):
[PPMI comparison skipped: 'NoneType' object is not iterable]


---
## Section 2: Dense Embeddings (25 points)

In this section, you will work with pre-trained dense word embeddings. Unlike the sparse, high-dimensional count-based vectors from Section 1, dense embeddings represent each word as a short (e.g., 300-dimensional), dense vector learned by a neural network.

The key insight is that dense embeddings can capture semantic relationships through vector arithmetic: the famous example $\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$ demonstrates that these vectors encode meaning in their geometry.

**Lectures:** Introducing Word2Vec through Analogies and Vector Arithmetic | **Textbook:** Chapter 5, Sections 5.5-5.7

### 2.1 Loading Pre-trained Embeddings (5 points)

*All 5 points are awarded for the writeup (Answer 2.1b below). Run the exploration cells, then include the output in your `hw04_writeup.pdf` as evidence for your written analysis.*

You loaded the embeddings in Section 0. Now explore them: examine the vocabulary, look at vector properties, and use gensim's `most_similar()` to find semantically related words.

In [ ]:
try:
    # Explore the pre-trained embeddings

    # Check if specific words are in the vocabulary
    test_words = ['king', 'queen', 'python', 'tensorflow', 'xyz123']
    for word in test_words:
        in_vocab = word in embeddings
        print(f"  '{word}' in vocabulary: {in_vocab}")

    # Look at a word vector
    king_vec = embeddings['king']
    print(f"\nVector for 'king':")
    print(f"  Shape: {king_vec.shape}")
    print(f"  Norm: {np.linalg.norm(king_vec):.4f}")
    print(f"  First 10 values: {king_vec[:10]}")
except Exception as e:
    print(f"[Embedding exploration skipped: {e}]")

[Embedding exploration skipped: argument of type 'NoneType' is not iterable]


In [ ]:
try:
    # Use gensim's built-in most_similar
    print("Most similar to 'king':")
    for word, sim in embeddings.most_similar('king', topn=10):
        print(f"  {word:<20} {sim:.4f}")

    print("\nMost similar to 'computer':")
    for word, sim in embeddings.most_similar('computer', topn=10):
        print(f"  {word:<20} {sim:.4f}")
except Exception as e:
    print(f"[Embedding similarity exploration skipped: {e}]")

Most similar to 'king':
[Embedding similarity exploration skipped: 'NoneType' object has no attribute 'most_similar']


### Answer 2.1b Polysemy and Static Embeddings (5 points) [WRITEUP]

A fundamental limitation of Word2Vec/GloVe is that each word gets **one vector**, regardless of context. But many words have multiple meanings (**polysemy**). The embedding for a polysemous word is effectively an average across all its senses.

Explore this by examining the nearest neighbors of at least **3 polysemous words**. Some good candidates:

| Word | Possible senses |
|------|----------------|
| `bank` | financial institution / river bank |
| `bug` | insect / software error / listening device |
| `apple` | fruit / company (try `Apple` with capital too!) |
| `bat` | flying mammal / sports equipment |
| `spring` | season / water source / coiled metal / to jump |
| `cell` | biology / prison / phone |

**[WRITEUP] (5 points):** For each word:
1. Print the 15 nearest neighbors using `embeddings.most_similar()`
2. Identify which senses are represented in the neighbors
3. Try the capitalization trick from the demo: does `apple` vs `Apple` give different neighbors?

Which sense dominates the nearest neighbors for each word? Why? What does this tell you about when static embeddings will work poorly for a downstream task? *(This limitation is what motivates contextual embeddings like BERT in Module 7.)*

> **Note on embedding model:** The capitalization trick (`apple` vs `Apple`) is most visible with the **300d Word2Vec Google News** model, which was trained on a large news corpus where capitalization carries strong signal. With the **100d GloVe** model, the effect may be subtler or absent. If you used the 100d model and the trick did not show a clear difference, report what you actually observe and note this limitation -- honest reporting of your results is expected.

In [ ]:
## YOUR CODE HERE
# Explore polysemous words. For each, print the 15 nearest neighbors
# and note which senses are represented.

try:
    polysemous_words = ['bank', 'bug', 'apple']  # Replace or add your own

    for word in polysemous_words:
        if word in embeddings:
            print(f"\nNearest neighbors of '{word}':")
            for neighbor, sim in embeddings.most_similar(word, topn=15):
                print(f"  {neighbor:<25} {sim:.4f}")
        else:
            print(f"'{word}' not in vocabulary")

    # Try capitalization: 'apple' (fruit) vs 'Apple' (company)
    print("\n--- Capitalization comparison ---")
    for word in ['apple', 'Apple']:
        if word in embeddings:
            neighbors = [w for w, _ in embeddings.most_similar(word, topn=10)]
            print(f"  '{word}' -> {neighbors}")
        else:
            print(f"  '{word}' not in vocabulary")
except Exception as e:
    print(f"[Polysemy exploration skipped: {e}]")

[Polysemy exploration skipped: argument of type 'NoneType' is not iterable]


### 2.2 Word Analogies (5 points)

*Point breakdown: 2 pts autograded (`solve_analogy`) | 3 pts writeup (Section 2.2b)*

Solve word analogies using vector arithmetic. The classic formulation:

**a : a\* :: b : b\*** -- "a is to a* as b is to b*"

is solved by finding:

$$b^* = \arg\max_{w \in V \setminus \{a, a^*, b\}} \cos(\vec{w}, \vec{b} - \vec{a} + \vec{a^*})$$

Implement this **from scratch** -- do not use gensim's built-in `most_similar(positive=..., negative=...)` method.

**Hint:** To iterate over all words in the embedding vocabulary, use `embeddings.index_to_key` (a list of all words in the model, ordered from most to least frequent).

**Lecture:** Analogies and Vector Arithmetic | **Textbook:** Chapter 5, Section 5.7

In [ ]:
def solve_analogy(a, a_star, b, embeddings, top_n=5):
    """Solve the analogy: a is to a_star as b is to ???

    Computes b* = argmax cos(w, b - a + a*) for all w in vocabulary,
    excluding a, a*, and b themselves.

    Do NOT use gensim's most_similar() -- implement from scratch using
    the embeddings vectors and your cosine_similarity function.

    Hint: use embeddings.index_to_key to get the list of all vocabulary words.

    Args:
        a: First word of the analogy (string).
        a_star: Second word (string). a is to a_star.
        b: Third word (string). b is to ???.
        embeddings: gensim KeyedVectors object.
        top_n: Number of candidate answers to return.

    Returns:
        A list of (word, similarity) tuples, sorted by similarity descending.
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Test analogies -- syntactic and semantic

    analogies = [
        # Semantic: capitals
        ('paris', 'france', 'tokyo', 'japan'),
        ('paris', 'france', 'berlin', 'germany'),
        # Semantic: gender
        ('man', 'woman', 'king', 'queen'),
        ('brother', 'sister', 'uncle', 'aunt'),
        # Syntactic: plural
        ('cat', 'cats', 'dog', 'dogs'),
        # Syntactic: tense
        ('walk', 'walked', 'swim', 'swam'),
    ]

    print(f"{'Analogy':<45} {'Top Answer':<15} {'Expected':<10} {'Correct?'}")
    print("-" * 80)

    for a, a_star, b, expected in analogies:
        results = solve_analogy(a, a_star, b, embeddings, top_n=5)
        top_answer = results[0][0] if results else 'N/A'
        correct = (top_answer.lower() == expected.lower())
        analogy_str = f"{a} : {a_star} :: {b} : ???"
        print(f"{analogy_str:<45} {top_answer:<15} {expected:<10} {'YES' if correct else 'no'}")
except Exception as e:
    print(f"[Analogy testing skipped: {e}]")

Analogy                                       Top Answer      Expected   Correct?
--------------------------------------------------------------------------------
paris : france :: tokyo : ???                 N/A             japan      no
paris : france :: berlin : ???                N/A             germany    no
man : woman :: king : ???                     N/A             queen      no
brother : sister :: uncle : ???               N/A             aunt       no
cat : cats :: dog : ???                       N/A             dogs       no
walk : walked :: swim : ???                   N/A             swam       no


In [ ]:
try:
    # Sanity checks for solve_analogy

    results = solve_analogy('man', 'woman', 'king', embeddings, top_n=5)

    assert isinstance(results, list), \
        f"solve_analogy should return a list, got {type(results)}"
    assert len(results) == 5, \
        f"Expected 5 results with top_n=5, got {len(results)}"
    assert all(isinstance(r, tuple) and len(r) == 2 for r in results), \
        "Each result should be a (word, similarity) tuple"

    # The answer should NOT include the input words
    result_words = [r[0] for r in results]
    for excluded in ['man', 'woman', 'king']:
        assert excluded not in result_words, \
            f"Input word '{excluded}' should be excluded from results"

    # Results should be sorted by similarity (descending)
    sims = [r[1] for r in results]
    assert sims == sorted(sims, reverse=True), \
        "Results should be sorted by similarity in descending order"

    print("Analogy function sanity checks passed!")
except Exception as e:
    print(f"[Analogy sanity checks skipped: {e}]")

[Analogy sanity checks skipped: solve_analogy should return a list, got <class 'NoneType'>]


### Answer 2.2b When Analogies Fail (3 points) [WRITEUP]

The analogies above work impressively well. But vector arithmetic doesn't always capture semantic relationships. In this exercise, you'll find cases where analogies **break down**.

**Task:** Construct **3 word analogies** where `solve_analogy` gives the **wrong** answer. For each:

1. Show the analogy (a : a* :: b : ???), the expected answer, and what your function actually returns
2. State the nature of the failure, such as those listed below, and explain **why** vector arithmetic fails for this relationship

**Common failure types:**
- **Morphological/negation mismatch**
- **Role asymmetry**
- **Polysemy/ambiguity**
- **Data sparsity**
- **World knowledge gap**

**Hints for finding failures:**
- Try **abstract or metaphorical relationships** (e.g., "problem : solution :: question : ???")
- Try **negation** (e.g., "happy : unhappy :: honest : ???")
- Try **asymmetric relationships** (the relation isn't parallel)
- Try analogies involving **rare or ambiguous words**
- Try relationships that require **world knowledge** beyond distributional patterns


In [ ]:
## YOUR CODE HERE
# Try different analogies and find 3 that fail.
# Use solve_analogy() and show what the model returns vs. what you expected.

try:
    # Example of testing an analogy:
    # results = solve_analogy('problem', 'solution', 'question', embeddings, top_n=5)
    # print(f"problem : solution :: question : ???")
    # print(f"  Expected: 'answer'")
    # print(f"  Got: {results[:3]}")
    pass
except Exception as e:
    print(f"[Analogy failure exploration skipped: {e}]")

### 2.3 Cosine Similarity & Evaluation (8 points)

*Point breakdown: 4 pts autograded (`evaluate_similarity`) | 4 pts writeup*

Compare count-based (PPMI) and dense (Word2Vec/GloVe) similarity scores on the same word pairs. Then evaluate both against human judgments using a subset of **SimLex-999**, a benchmark dataset of word pairs with human-annotated similarity scores.

`compute_similarity_scores` (in the next cell) is provided for you -- your task is to implement `evaluate_similarity`, which computes the Kendall's Tau rank correlation between model scores and human scores.

Use **Kendall's Tau** ($\tau$) to measure the rank correlation between your model's similarity rankings and the human rankings. Kendall's Tau ranges from $-1$ (perfectly reversed rankings) through $0$ (no correlation) to $+1$ (perfect agreement).

**Lecture:** Evaluating Embeddings | **Textbook:** Chapter 5, Sections 5.4 and 5.9

In [ ]:
########## DO NOT MODIFY ##########

SIMLEX_PATH = 'data/simlex999.txt'

def load_simlex(filepath, vocab_set=None, embeddings=None, max_pairs=None):
    """Load SimLex-999 word pairs, filtering to words in our vocabularies.

    Returns a list of (word1, word2, human_score) tuples.
    Only includes pairs where both words are in vocab_set AND embeddings.
    """
    pairs = []
    with open(filepath, encoding='utf-8') as f:
        header = f.readline()  # skip header
        for line in f:
            parts = line.strip().split('\t')
            w1, w2 = parts[0].lower(), parts[1].lower()
            score = float(parts[3])  # SimLex999 column
            # Filter: both words must be in both vocabularies
            in_sparse = (vocab_set is None) or (w1 in vocab_set and w2 in vocab_set)
            in_dense = (embeddings is None) or (w1 in embeddings and w2 in embeddings)
            if in_sparse and in_dense:
                pairs.append((w1, w2, score))
            if max_pairs and len(pairs) >= max_pairs:
                break
    return pairs


try:
    assert os.path.exists(SIMLEX_PATH), f"Missing data file: {SIMLEX_PATH}"

    # Load SimLex pairs that exist in BOTH our Shakespeare vocab AND the embeddings
    vocab_set = set(vocab)
    simlex_pairs = load_simlex(SIMLEX_PATH, vocab_set=vocab_set, embeddings=embeddings)
    print(f"Loaded {len(simlex_pairs)} SimLex-999 pairs present in both vocabularies.")
    if len(simlex_pairs) < 20:
        print("WARNING: Very few overlapping pairs. Consider using a larger Shakespeare vocabulary")
        print("         (reduce MIN_COUNT in Section 0) or loading more SimLex pairs.")
    print(f"Sample pairs: {simlex_pairs[:5]}")

except Exception as e:
    print(f"[SimLex data loading skipped: {e}]")
    simlex_pairs = []

Loaded 53 SimLex-999 pairs present in both vocabularies.
Sample pairs: [('old', 'new', 1.58), ('happy', 'mad', 0.95), ('certain', 'sure', 8.42), ('happy', 'young', 2.0), ('wife', 'husband', 2.3)]


In [ ]:
def compute_similarity_scores(pairs, method, **kwargs):
    """Compute similarity scores for a list of word pairs.

    Args:
        pairs: List of (word1, word2, human_score) tuples.
        method: One of 'ppmi', 'dense'.
        **kwargs: Additional arguments:
            For 'ppmi': ppmi_matrix, word_to_idx
            For 'dense': embeddings

    Returns:
        A list of model similarity scores (same length as pairs).
    """
    scores = []
    for w1, w2, _ in pairs:
        if method == 'ppmi':
            ppmi_mat = kwargs['ppmi_matrix']
            w2i = kwargs['word_to_idx']
            v1 = ppmi_mat[w2i[w1]]
            v2 = ppmi_mat[w2i[w2]]
            scores.append(cosine_similarity(v1, v2))
        elif method == 'dense':
            emb = kwargs['embeddings']
            v1 = emb[w1]
            v2 = emb[w2]
            scores.append(cosine_similarity(v1, v2))
        else:
            raise ValueError(f"Unknown method: {method}")
    return scores

In [ ]:
from scipy.stats import kendalltau

def evaluate_similarity(pairs, method, **kwargs):
    """Evaluate word similarity against human judgments using Kendall's Tau.

    Args:
        pairs: List of (word1, word2, human_score) tuples.
        method: One of 'ppmi', 'dense'.
        **kwargs: Passed to compute_similarity_scores.

    Returns:
        Kendall's Tau correlation (float).
    """
    ## YOUR CODE HERE
    # 1. Extract human scores from the pairs
    # 2. Compute model scores using compute_similarity_scores
    # 3. Compute and return Kendall's Tau between human and model scores
    pass

In [ ]:
try:
    # Compare count-based vs dense similarity

    tau_ppmi = evaluate_similarity(simlex_pairs, 'ppmi',
                                    ppmi_matrix=ppmi_matrix, word_to_idx=word_to_idx)
    tau_dense = evaluate_similarity(simlex_pairs, 'dense', embeddings=embeddings)

    print(f"Kendall's Tau on SimLex-999 ({len(simlex_pairs)} pairs):")
    print(f"  PPMI (count-based):   {tau_ppmi:.4f}")
    print(f"  Dense (Word2Vec):     {tau_dense:.4f}")
    print(f"  Difference:           {tau_dense - tau_ppmi:+.4f}")
except Exception as e:
    print(f"[Similarity evaluation skipped: {e}]")

Kendall's Tau on SimLex-999 (53 pairs):
[Similarity evaluation skipped: unsupported format string passed to NoneType.__format__]


In [ ]:
try:
    # Show individual pair comparisons
    ppmi_scores = compute_similarity_scores(simlex_pairs, 'ppmi',
                                             ppmi_matrix=ppmi_matrix, word_to_idx=word_to_idx)
    dense_scores = compute_similarity_scores(simlex_pairs, 'dense', embeddings=embeddings)
    human_scores = [p[2] for p in simlex_pairs]

    print(f"{'Word Pair':<25} {'Human':>8} {'PPMI':>8} {'Dense':>8}")
    print("-" * 55)
    for i, (w1, w2, hs) in enumerate(simlex_pairs[:15]):
        print(f"{w1 + ' - ' + w2:<25} {hs:>8.2f} {ppmi_scores[i]:>8.4f} {dense_scores[i]:>8.4f}")
except Exception as e:
    print(f"[Pair comparison skipped: {e}]")

[Pair comparison skipped: 'NoneType' object is not subscriptable]


### [WRITEUP] Answer 2.3 (4 points)

How do count-based (PPMI) and dense (Word2Vec/GloVe) embeddings compare on the SimLex-999 word similarity task?

1. Which representation achieves higher Kendall's Tau? By how much?
2. Give at least 2 specific word pairs where the representations disagree significantly with each other. For each, explain why one model might be better.
3. Why might dense embeddings outperform sparse count-based embeddings on word similarity? Consider at least two of the following: (a) the effect of dimensionality reduction, (b) the size/diversity of training data, (c) the objective function used to learn the embeddings.

Reference actual Tau values and word pairs from your experiments.

> **Note on embedding model:** Your Kendall's Tau values will depend on which model you loaded. The **300d Word2Vec** model typically achieves higher correlation with human judgments than the **100d GloVe** model, because it was trained on far more data. Whatever model you used, report those exact Tau values and name the model (e.g., `glove-wiki-gigaword-100` or `word2vec-google-news-300`) in your write-up.

### 2.4 Bias in Embeddings (7 points)

*Point breakdown: 4 pts autograded (`measure_gender_bias`) | 3 pts writeup*

Word embeddings trained on large text corpora absorb and amplify societal biases present in the training data. In this section, you will measure gender bias in the pre-trained embeddings by comparing occupation words to gender direction vectors.

**Lecture:** Bias in Embeddings | **Textbook:** Chapter 5, Section 5.8

**Reference:** Bolukbasi et al. (2016), "Man is to Computer Programmer as Woman is to Homemaker?"

In [ ]:
def measure_gender_bias(occupation_words, embeddings, male_words=None, female_words=None):
    """Measure gender bias for a list of occupation words.

    For each occupation word, compute:
        bias = cos(occupation, she_vec) - cos(occupation, he_vec)

    where she_vec and he_vec are the averages of the female and male
    word vectors respectively.

    Positive bias = associated more with female
    Negative bias = associated more with male

    Args:
        occupation_words: List of occupation strings.
        embeddings: gensim KeyedVectors object.
        male_words: List of male-associated words (default: ['he', 'man', 'boy', 'father', 'son']).
        female_words: List of female-associated words (default: ['she', 'woman', 'girl', 'mother', 'daughter']).

    Returns:
        A list of (occupation, bias_score) tuples, sorted by bias_score.
    """
    if male_words is None:
        male_words = ['he', 'man', 'boy', 'father', 'son']
    if female_words is None:
        female_words = ['she', 'woman', 'girl', 'mother', 'daughter']

    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Measure gender bias across occupations
    occupations = [
        'doctor', 'nurse', 'engineer', 'teacher', 'professor', 'scientist',
        'lawyer', 'secretary', 'programmer', 'homemaker', 'architect',
        'librarian', 'surgeon', 'receptionist', 'mechanic', 'pilot',
        'accountant', 'plumber', 'CEO', 'nanny', 'carpenter', 'dancer',
        'firefighter', 'housekeeper', 'electrician', 'therapist'
    ]

    # Filter to words in the embedding vocabulary
    occupations = [w for w in occupations if w in embeddings]

    bias_results = measure_gender_bias(occupations, embeddings)

    assert len(bias_results) > 0, "Should have at least some bias results"
    assert all(isinstance(r, tuple) and len(r) == 2 for r in bias_results), \
        "Each result should be a (occupation, bias_score) tuple"

    print(f"Gender Bias Scores (positive = female-associated, negative = male-associated)")
    print(f"{'Occupation':<20} {'Bias Score':>12} {'Direction':>10}")
    print("-" * 45)
    for occ, bias in bias_results:
        direction = 'FEMALE' if bias > 0.05 else ('MALE' if bias < -0.05 else 'neutral')
        print(f"{occ:<20} {bias:>12.4f} {direction:>10}")
except Exception as e:
    print(f"[Bias analysis skipped: {e}]")
    bias_results = []

[Bias analysis skipped: argument of type 'NoneType' is not iterable]


In [ ]:
try:
    # Visualize the bias spectrum
    if bias_results:
        names = [r[0] for r in bias_results]
        scores = [r[1] for r in bias_results]

        colors = ['blue' if s < -0.05 else ('red' if s > 0.05 else 'gray') for s in scores]

        plt.figure(figsize=(10, 8))
        plt.barh(range(len(names)), scores, color=colors)
        plt.yticks(range(len(names)), names)
        plt.xlabel('Gender Bias Score (negative = male, positive = female)')
        plt.title('Gender Bias in Word Embeddings by Occupation')
        plt.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"[Bias visualization skipped: {e}]")

### [WRITEUP] Answer 2.4 (3 points)

1. What gender biases did you find? Identify at least **3 stereotypical associations** the embeddings encode (e.g., specific occupations strongly associated with one gender).
2. Where does this bias originate? Consider the training data AND either the distributional hypothesis or the objective function.
3. What are the implications for NLP systems that use these embeddings? Consider one of the following example applications (resume screening, machine translation, search engines), or your own valid one.

---
## Section 3: Word Sense Clustering (25 points)

A word like "bank" can mean a financial institution or the side of a river. In this section, you will use **K-means clustering** to discover word senses from paraphrases.

**Task:** Given a target word and a set of its paraphrases, cluster the paraphrases so that each cluster corresponds to a different sense of the target word. For example, paraphrases of "bank" might cluster into:
- Cluster 1: {"financial institution", "credit union", "savings and loan"} (financial sense)
- Cluster 2: {"shore", "riverside", "embankment"} (river sense)

You will represent paraphrases using both sparse (co-occurrence) and dense (embedding) vectors, cluster them, and evaluate using the **paired F-score**.

**Lectures:** Word Representation by Context, Introducing Word2Vec | **Textbook:** Chapter 5, Sections 5.5-5.6

In [ ]:
########## DO NOT MODIFY ##########

def load_cooccurrence_vectors(paraphrases, cooc_dir):
    """Load pre-computed co-occurrence vectors for paraphrases.

    Each paraphrase has a sparse co-occurrence vector stored as a text file
    in the cooccurrence directory.

    Args:
        paraphrases: List of paraphrase strings.
        cooc_dir: Path to directory containing co-occurrence vector files.

    Returns:
        A numpy array of shape (len(paraphrases), d) where d is the
        dimensionality of the co-occurrence vectors.
    """
    vectors = []
    for pp in paraphrases:
        # Filename is the paraphrase with spaces replaced by underscores
        fname = pp.replace(' ', '_') + '.txt'
        fpath = os.path.join(cooc_dir, fname)
        if os.path.exists(fpath):
            vec = np.loadtxt(fpath)
        else:
            # If no pre-computed vector, use a zero vector
            vec = np.zeros(500)  # default dimensionality
        vectors.append(vec)
    return np.array(vectors)


def get_embedding_vectors(paraphrases, embeddings):
    """Get dense embedding vectors for paraphrases.

    For multi-word paraphrases, average the word vectors.
    For words not in the embedding vocabulary, use a zero vector.

    Args:
        paraphrases: List of paraphrase strings.
        embeddings: gensim KeyedVectors object.

    Returns:
        A numpy array of shape (len(paraphrases), embedding_dim).
    """
    vectors = []
    dim = embeddings.vector_size
    for pp in paraphrases:
        words = pp.split()
        word_vecs = [embeddings[w] for w in words if w in embeddings]
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(np.zeros(dim))
    return np.array(vectors)


def paired_fscore(predicted_labels, gold_labels):
    """Compute paired F-score for clustering evaluation.

    The paired F-score considers all pairs of items:
    - TP: pairs in same cluster in both predicted and gold
    - FP: pairs in same predicted cluster but different gold clusters
    - FN: pairs in same gold cluster but different predicted clusters

    F-score = 2 * P * R / (P + R) where P = TP/(TP+FP), R = TP/(TP+FN)

    Args:
        predicted_labels: List of predicted cluster assignments.
        gold_labels: List of gold-standard cluster assignments.

    Returns:
        Paired F-score (float in [0, 1]).
    """
    n = len(predicted_labels)
    assert len(gold_labels) == n

    tp = fp = fn = 0
    for i in range(n):
        for j in range(i + 1, n):
            same_pred = (predicted_labels[i] == predicted_labels[j])
            same_gold = (gold_labels[i] == gold_labels[j])
            if same_pred and same_gold:
                tp += 1
            elif same_pred and not same_gold:
                fp += 1
            elif not same_pred and same_gold:
                fn += 1

    if tp == 0:
        return 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


print("Clustering helper functions loaded.")

Clustering helper functions loaded.


### 3.1 Sparse Representation Clustering (8 points)

Cluster paraphrases using pre-computed **co-occurrence vectors** (sparse representation). For each target word in the dev set:

1. Load the co-occurrence vectors for its paraphrases
2. Apply K-means clustering with K = the gold-standard number of senses
3. Evaluate using the paired F-score

In [ ]:
def cluster_sparse(data, cooc_dir):
    """Cluster paraphrases using sparse co-occurrence vectors.

    For each target word entry in data:
    1. Load co-occurrence vectors for the paraphrases
    2. Run K-means with K = entry['k'] (gold-standard number of senses)
    3. Compute paired F-score against gold labels

    Use random_state=42 in KMeans for reproducibility.

    Args:
        data: List of dicts from load_cluster_data().
        cooc_dir: Path to co-occurrence vector directory.

    Returns:
        A list of dicts, each with:
            'target': target word
            'k': number of clusters
            'fscore': paired F-score
            'predicted_labels': list of predicted cluster assignments
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Run sparse clustering on dev set
    sparse_results = cluster_sparse(dev_data, CLUSTER_COOCCURRENCE_DIR)

    assert len(sparse_results) == len(dev_data), \
        f"Expected {len(dev_data)} results, got {len(sparse_results)}"

    sparse_fscores = [r['fscore'] for r in sparse_results]
    avg_sparse_f = np.mean(sparse_fscores)

    print(f"Sparse clustering results (dev set):")
    print(f"{'Target':<20} {'K':>3} {'F-score':>8}")
    print("-" * 35)
    for r in sparse_results:
        print(f"{r['target']:<20} {r['k']:>3} {r['fscore']:>8.4f}")
    print(f"\nAverage F-score (sparse): {avg_sparse_f:.4f}")
except Exception as e:
    print(f"[Sparse clustering skipped: {e}]")
    sparse_results = []
    avg_sparse_f = 0.0

[Sparse clustering skipped: object of type 'NoneType' has no len()]


### 3.2 Dense Representation Clustering (8 points)

Now cluster the same paraphrases using **dense embedding vectors** (Word2Vec/GloVe). Compare performance to sparse clustering.

In [ ]:
def cluster_dense(data, embeddings):
    """Cluster paraphrases using dense embedding vectors.

    For each target word entry in data:
    1. Get embedding vectors for the paraphrases (using get_embedding_vectors)
    2. Run K-means with K = entry['k'] (gold-standard number of senses)
    3. Compute paired F-score against gold labels

    Use random_state=42 in KMeans for reproducibility.

    Args:
        data: List of dicts from load_cluster_data().
        embeddings: gensim KeyedVectors object.

    Returns:
        Same format as cluster_sparse.
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Run dense clustering on dev set
    dense_results = cluster_dense(dev_data, embeddings)

    assert len(dense_results) == len(dev_data)

    dense_fscores = [r['fscore'] for r in dense_results]
    avg_dense_f = np.mean(dense_fscores)

    # Side-by-side comparison
    print(f"{'Target':<20} {'K':>3} {'Sparse F':>10} {'Dense F':>10} {'Winner':>8}")
    print("-" * 55)
    for s, d in zip(sparse_results, dense_results):
        winner = 'sparse' if s['fscore'] > d['fscore'] else ('dense' if d['fscore'] > s['fscore'] else 'tie')
        print(f"{s['target']:<20} {s['k']:>3} {s['fscore']:>10.4f} {d['fscore']:>10.4f} {winner:>8}")

    print(f"\nAverage F-score (sparse): {avg_sparse_f:.4f}")
    print(f"Average F-score (dense):  {avg_dense_f:.4f}")
except Exception as e:
    print(f"[Dense clustering skipped: {e}]")
    dense_results = []
    avg_dense_f = 0.0

[Dense clustering skipped: object of type 'NoneType' has no len()]


### 3.3 Automatic K Selection (9 points)

*Point breakdown: 5 pts autograded (`cluster_auto_k`) | 4 pts writeup (Answer 3.3 below)*

In practice, we often don't know the true number of senses (K) for a word. Use the **silhouette score** to automatically select K.

The silhouette score measures how well each point fits in its assigned cluster vs. the nearest alternative cluster. It ranges from $-1$ (bad) to $+1$ (good). The optimal K maximizes the average silhouette score.

For each target word, search K from 2 to $\min(n_{\text{paraphrases}} - 1, 10)$ and select the K with the highest silhouette score.

In [ ]:
def cluster_auto_k(data, embeddings):
    """Cluster paraphrases with automatic K selection using silhouette score.

    For each target word:
    1. Get embedding vectors for the paraphrases
    2. For K in range(2, min(n_paraphrases - 1, 10) + 1):
        a. Run K-means with this K (random_state=42)
        b. Compute silhouette score
    3. Select the K with the highest silhouette score
    4. Compute paired F-score with gold labels

    Args:
        data: List of dicts from load_cluster_data().
        embeddings: gensim KeyedVectors object.

    Returns:
        A list of dicts, each with:
            'target': target word
            'gold_k': gold-standard K
            'predicted_k': automatically selected K
            'fscore': paired F-score
            'predicted_labels': list of predicted cluster assignments
    """
    ## YOUR CODE HERE
    pass

In [ ]:
try:
    # Run automatic K selection
    auto_results = cluster_auto_k(dev_data, embeddings)

    assert len(auto_results) == len(dev_data)

    auto_fscores = [r['fscore'] for r in auto_results]
    avg_auto_f = np.mean(auto_fscores)

    # Compare predicted K to gold K
    k_matches = sum(1 for r in auto_results if r['predicted_k'] == r['gold_k'])

    print(f"Automatic K selection results:")
    print(f"{'Target':<20} {'Gold K':>7} {'Pred K':>7} {'F-score':>8} {'K Match?':>9}")
    print("-" * 55)
    for r in auto_results:
        match = 'YES' if r['predicted_k'] == r['gold_k'] else 'no'
        print(f"{r['target']:<20} {r['gold_k']:>7} {r['predicted_k']:>7} {r['fscore']:>8.4f} {match:>9}")

    print(f"\nK matches: {k_matches}/{len(auto_results)}")
    print(f"Average F-score (auto K):  {avg_auto_f:.4f}")
    print(f"Average F-score (gold K):  {avg_dense_f:.4f}")
except Exception as e:
    print(f"[Auto-K clustering skipped: {e}]")

[Auto-K clustering skipped: object of type 'NoneType' has no len()]


### [WRITEUP] Answer 3.3 (4 points)

Compare the three clustering approaches you implemented (sparse co-occurrence, dense embeddings with gold K, and dense embeddings with automatic K):

1. **Sparse vs. Dense:** Which representation -- sparse co-occurrence vectors or dense embeddings -- works better for word sense clustering on the dev set? Report the average F-scores for both and explain why one outperforms the other.
2. **Automatic K selection:** How does the silhouette-score method for selecting K compare to using the gold-standard K? For how many target words did your predicted K match the gold K? When does automatic K selection work well, and when does it fail? *(Hint: "when" refers to specific target words in the dev set -- look at the words where auto-K got the right answer vs. where it didn't -- cite specific words from the dev set. What makes some words easier to cluster automatically? Consider the number of paraphrases available, how semantically distinct the senses are from each other, and whether the gold K is 2 vs. larger.)*
3. **Limitations:** What are the limitations of representing multi-word paraphrases by averaging their word vectors? Suggest one alternative approach that might better capture the meaning of a multi-word phrase.

> **Note on embedding model:** Clustering F-scores and automatic K selection results may vary depending on whether you used the **100d GloVe** or **300d Word2Vec** model. Report results from whichever model you ran, and note which model you used in your write-up.

---
## Section 4: Analysis & Reflection (25 points)

This section is **entirely writeup** -- no code to implement. Answer the questions below in your `hw04_writeup.pdf`. Use evidence from your experiments in Sections 1-3 to support your arguments.

| Question | Topic | Points |
|----------|-------|--------|
| 4.1 | Count-Based vs Dense Representations | 5 pts |
| 4.2 | Shakespeare Play Analysis | 15 pts |
| 4.3 | From Embeddings to Language Models | 5 pts |

### [WRITEUP] Answer 4.1 (5 points): Count-Based vs Dense Representations

You have now worked extensively with both paradigms. Compare them across three dimensions, using **specific examples from your experiments** (not just abstract claims):

**(a) Interpretability:** Given a word's count-based vector (e.g., its PPMI row), can you understand *why* the model considers two words similar? Can you do the same with a dense embedding vector? Which is more interpretable, and what is the cost of that interpretability? *(Hint: think about what the individual dimensions in each vector representation mean.)*

**(b) Coverage of unseen words:** A count-based representation only covers words that appeared in your Shakespeare training corpus, so a word absent from the vocabulary has no row in your matrix at all. Dense pre-trained embeddings cover far more words but still have a fixed vocabulary. Pick a word that you'd expect to be in pre-trained word2vec/GloVe but missing from your Shakespeare PPMI vocabulary (a modern technical term, brand name, or proper noun, say). Explain what this means for each representation: which queries can each one answer, and which fail? How do subword tokenizers (the BPE you implemented in HW2) sidestep this limitation entirely?

**(c) Computational requirements:** Compare the storage, computation time, and data requirements of the two approaches. Which is more practical for a real-world application?

### [WRITEUP] Answer 4.2 (15 points): Shakespeare Play Analysis

Choose **ONE** of the following analyses and write a thorough response. Use your term-document matrix, TF-IDF weights, and/or play similarity rankings as evidence.

**(a) Comedies vs. Tragedies:** Can you distinguish Shakespeare's comedies from his tragedies using the vector representations you built? Look at the play similarity rankings from Section 1.1. Do comedies cluster together? Do tragedies? Show specific evidence.

**(b) Character Clustering:** Which character names (e.g., "hamlet", "juliet", "prospero") cluster together in the word embedding space? What does this reveal about how Shakespeare used these characters? Look at both the count-based and dense representations.

**(c) The Unusual Play:** Which Shakespeare play is most "unusual" -- i.e., least similar to all other plays on average? Use your cosine similarity rankings to identify it, and examine its TF-IDF-weighted discriminative words to explain *why* it stands out.

**Note**: You may want to comment out your experimental code from this section, in your `hw04.py` file, if you experience issues with the autograder.

In [ ]:
# (Optional) This is the place to put experimental code for Section 4.2 if needed.
# For example, to generate analysis, additional data or visualizations to support your write-up.
if __name__ == "__main__":
    pass

### [WRITEUP] Answer 4.3 (5 points): From Embeddings to Language Models

Word embeddings represent **static** word meaning -- each word gets exactly one vector regardless of the context it appears in.

1. What is the fundamental limitation of this approach? Give a **specific example** of a word where a single vector fails to capture its meaning (think about polysemy, metaphor, or domain-specific usage).
2. How do you think this could be addressed? (Hint: you will see the answer in Module 07 when we study BERT and contextual embeddings.)
3. Connect this to the Context Dependence lecture: what did the lecture say about the limitations of static word embeddings?

**Note:** You are not expected to know the details of contextual embeddings yet -- just reason about what would need to change.

---
## Submission Checklist

Before submitting, verify:

- [ ] All code cells run without errors (Kernel -> Restart & Run All)
- [ ] All assertion/sanity checks pass
- [ ] `hw04_writeup.pdf` contains answers to all **[WRITEUP]** questions:
  - **Answer 2.1b** (5 pts): Polysemy exploration — include `most_similar()` output for 3 words, dominant senses, capitalization effect
  - **Answer 2.2b** (3 pts): Analogy failures (3 failed analogies with explanations)
  - **Answer 2.3** (4 pts): Count-based vs dense on SimLex-999
  - **Answer 2.4** (3 pts): Gender bias analysis
  - **Answer 3.3** (4 pts): Sparse vs dense clustering; automatic K selection; limitations
  - **Answer 4.1** (5 pts): Count-based vs dense representations (interpretability, coverage, computation)
  - **Answer 4.2** (15 pts): Shakespeare play analysis (choose ONE of a/b/c)
  - **Answer 4.3** (5 pts): From embeddings to language models (polysemy limitation)
- [ ] Files are named exactly: `hw04.ipynb`, `hw04.py`, `hw04_writeup.pdf`

### Point Summary

| Section | Topic | Autograded | Writeup | Total |
|---------|-------|-----------|---------|-------|
| 1 | Count-Based Representations | 25 pts | -- | 25 pts |
| 2 | Dense Embeddings | 10 pts | 15 pts | 25 pts |
| 3 | Word Sense Clustering | 21 pts | 4 pts | 25 pts |
| 4 | Analysis & Reflection | -- | 25 pts | 25 pts |
| **Total** | | **56 pts** | **44 pts** | **100 pts** |

### Dependencies

This notebook requires: `numpy`, `scikit-learn`, `gensim`, `matplotlib`, `scipy`

```bash
pip install numpy scikit-learn gensim matplotlib scipy
```